# Gemini Search to BigQuery Data Pipeline

This notebook demonstrates a pipeline that uses Gemini to perform a search (via a custom search tool), extract structured information from the search results, and then load this structured data into a BigQuery table.

## Prerequisites

Before running this notebook, ensure you have the following:

1.  **Google Cloud Project**: 
    *   Vertex AI API enabled.
    *   BigQuery API enabled.
2.  **Google Custom Search API**:
    *   API enabled in your Google Cloud Project.
    *   An API Key.
    *   A Search Engine ID configured to search the desired websites (e.g., the entire web).
3.  **Authenticated Environment**: You need to be authenticated to Google Cloud. The simplest way for a local environment is to run:
    ```bash
    gcloud auth application-default login
    ```

## Placeholder Values to Replace

You **MUST** replace the placeholder values in the following cells for this notebook to work correctly:

*   **Cell 2 (Authentication and Client Initialization):**
    *   `PROJECT_ID = "your-gcp-project-id"`  <--- **REPLACE** with your Google Cloud Project ID.
    *   `LOCATION = "us-central1"`       <--- **REPLACE** with your desired GCP region (e.g., "us-central1").
*   **Cell 3 (Implement Search Functionality):**
    *   `GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY"` <--- **REPLACE** with your Google Custom Search API Key.
    *   `SEARCH_ENGINE_ID = "YOUR_SEARCH_ENGINE_ID"` <--- **REPLACE** with your Search Engine ID.
    *   `USER_SEARCH_QUERY = "..."` <--- **MODIFY** this with your desired search query (optional, an example is provided).
*   **Cell 4 (Data Extraction and Structuring):**
    *   `BIGQUERY_DATASET_ID = "gemini_bq_dataset"` <--- **CONFIRM OR REPLACE** with your preferred BigQuery Dataset ID.
    *   `BIGQUERY_TABLE_ID = "quantum_breakthroughs"` <--- **CONFIRM OR REPLACE** with your preferred BigQuery Table ID.
    *   The `BIGQUERY_TABLE_SCHEMA` is defined here; ensure it matches the data you intend to extract.

Carefully review each cell for any `TODO` or `<--- REPLACE` comments.

In [ ]:
# CELL 1: Install necessary libraries
# This cell installs the required Python packages from PyPI.
# - google-cloud-aiplatform: For Vertex AI services, including Gemini.
# - google-cloud-bigquery: For interacting with BigQuery.
# - google-api-python-client: For calling Google APIs, used here for the Custom Search API.
# - ipykernel: Required for running this notebook in Jupyter environments.

!pip install --upgrade google-cloud-aiplatform google-cloud-bigquery google-api-python-client ipykernel

In [ ]:
# CELL 2: Import required libraries and Initialize Clients

# Python standard libraries
import os # For environment variables, though not directly used for critical paths here.
import json # For handling JSON data, especially from Gemini's structured output.

# Google Cloud libraries
from google.cloud import aiplatform
from google.cloud import bigquery
from googleapiclient.discovery import build # For Google Custom Search API client

# Vertex AI specific imports for Gemini
from vertexai.generative_models import GenerativeModel, Tool, Part, FunctionDeclaration

# --- Configuration: Project ID and Location ---
# TODO: Please replace these with your actual project ID and desired location.
# These values are crucial for initializing Google Cloud services.

PROJECT_ID = "your-gcp-project-id"  # <--- REPLACE WITH YOUR GCP PROJECT ID
LOCATION = "us-central1"          # <--- REPLACE WITH YOUR GCP REGION (e.g., "us-central1", "europe-west1")

# --- Initialize Vertex AI SDK ---
# This step initializes the Vertex AI SDK, allowing communication with Vertex AI services.
# It requires the Project ID and Location defined above.
print(f"Initializing Vertex AI for project: {PROJECT_ID} in location: {LOCATION}")
try:
    aiplatform.init(project=PROJECT_ID, location=LOCATION)
    print("Vertex AI SDK initialized successfully.")
except Exception as e:
    print(f"Error initializing Vertex AI SDK: {e}")
    print("Please ensure your Project ID and Location are correct and you have authenticated.")

# --- Initialize BigQuery Client ---
# This step creates a BigQuery client, which is used to interact with BigQuery services
# (e.g., creating datasets/tables, inserting data).
print(f"Initializing BigQuery client for project: {PROJECT_ID}")
try:
    bq_client = bigquery.Client(project=PROJECT_ID)
    print("BigQuery client initialized successfully.")
    # Example: List datasets to verify BigQuery connection (optional, uncomment to test)
    # print("Attempting to list BigQuery datasets...")
    # datasets = list(bq_client.list_datasets(max_results=5)) # Limiting results
    # print(f"Successfully fetched {len(datasets)} datasets (up to 5 shown). Example: {datasets[0].dataset_id if datasets else 'No datasets found.'}")
except Exception as e:
    print(f"Error initializing BigQuery client or listing datasets: {e}")
    print("Please ensure your Project ID is correct and the BigQuery API is enabled.")

In [ ]:
# CELL 3: Implement Search Functionality using Gemini Tool Use

# --- Configuration: Google Custom Search API ---
# TODO: Replace with your Google Custom Search API key and Search Engine ID.
# These are essential for the perform_google_search function to work.
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY"      # <--- REPLACE with your API Key
SEARCH_ENGINE_ID = "YOUR_SEARCH_ENGINE_ID"  # <--- REPLACE with your Search Engine ID

# --- Define the Google Search Function for Gemini ---
# This function is what Gemini will 'call' when it needs to perform a search.
# It uses the Google Custom Search API.
def perform_google_search(search_query: str, num_results: int = 3):
    """Performs a Google search using the Custom Search API.
    Args:
        search_query: The query to search for.
        num_results: The number of search results to return (default: 3).
    Returns:
        A list of search results (items from the API response) or an empty list if an error occurs.
    """
    print(f"Executing Google Search for: '{search_query}' (max {num_results} results)")
    try:
        # Build the Custom Search service object
        service = build("customsearch", "v1", developerKey=GOOGLE_API_KEY)
        # Execute the search request
        result = service.cse().list(
            q=search_query,
            cx=SEARCH_ENGINE_ID,
            num=num_results
        ).execute()
        # Return the 'items' (search results) or an empty list if not found
        return result.get('items', [])
    except Exception as e:
        print(f"Error during Google Search API call: {e}")
        print("Please ensure your GOOGLE_API_KEY and SEARCH_ENGINE_ID are correct and the API is enabled.")
        return [] # Return empty list on error

# --- Define the Search Tool for Gemini ---
# This tells Gemini about the 'perform_google_search' function, its purpose, and its parameters.
google_search_tool = Tool(
    function_declarations=[
        FunctionDeclaration(
            name="perform_google_search",
            description="Performs a Google search to find information on a given topic. Returns a list of search results including snippets and links.",
            parameters={
                "type": "object",
                "properties": {
                    "search_query": {
                        "type": "string",
                        "description": "The search query or topic to look up."
                    },
                    "num_results": {
                        "type": "integer",
                        "description": "The maximum number of search results to return (e.g., 2-5). Default is 3."
                    }
                },
                "required": ["search_query"] # 'search_query' is a mandatory parameter
            }
        )
    ]
)

# --- Initialize the Generative Model with the Search Tool ---
# We use a Gemini model version that supports tool use (e.g., "gemini-1.5-pro-preview-0409" or "gemini-1.0-pro").
print("Initializing Gemini model with search tool...")
try:
    # Using a specific version known for robust tool use. You can adapt this to other compatible models.
    search_model = GenerativeModel(
        "gemini-1.5-pro-preview-0409", 
        tools=[google_search_tool]
    )
    print("Gemini model for search initialized successfully.")
except Exception as e:
    print(f"Error initializing Gemini model for search: {e}")

# --- Define User's Search Query and Prompt Gemini ---
# TODO: You can change this query to search for different information.
USER_SEARCH_QUERY = "Latest advancements in AI for drug discovery" 

print(f"\nPreparing to search for: '{USER_SEARCH_QUERY}' using Gemini and the Google Search tool...")

# This prompt instructs Gemini on what to do: use the search tool for the given query.
gemini_prompt_for_search = f"""
Please find information about: {USER_SEARCH_QUERY}.
Use the 'perform_google_search' tool to get this information. You can decide on the number of results, but keep it small (e.g., 3 or 4).
"""

search_results_from_tool = [] # Initialize an empty list to store results

try:
    if 'search_model' in globals(): # Check if model was initialized
        # Send the prompt to Gemini
        print("Sending prompt to Gemini to request tool execution...")
        response_from_gemini = search_model.generate_content(gemini_prompt_for_search)
        # print("Gemini's raw response for tool call:", response_from_gemini) # For debugging

        # Check if Gemini wants to call the tool
        if response_from_gemini.candidates and response_from_gemini.candidates[0].finish_reason == "TOOL_EXECUTION":
            function_call_request = response_from_gemini.candidates[0].content.parts[0].function_call
            if function_call_request.name == "perform_google_search":
                args = function_call_request.args
                print(f"Gemini requested to call 'perform_google_search' with arguments: {args}")
                
                # Actually call the Python function with arguments provided by Gemini
                search_results_from_tool = perform_google_search(
                    search_query=args['search_query'],
                    num_results=args.get('num_results', 3) # Use Gemini's num_results or default to 3
                )
            
                print("\n--- Search Results from Tool ---")
                if search_results_from_tool:
                    for i, item in enumerate(search_results_from_tool):
                        print(f"Result {i+1}:")
                        print(f"  Title: {item.get('title')}")
                        print(f"  Link: {item.get('link')}")
                        print(f"  Snippet: {item.get('snippet')}")
                        print("-" * 30)
                else:
                    print("No search results were returned from the tool, or an error occurred during the search.")
        else:
            print("Gemini did not request a tool execution. Its response was:")
            if response_from_gemini.candidates and response_from_gemini.candidates[0].content.parts:
                 print(response_from_gemini.candidates[0].content.parts[0].text)
            else:
                 print(response_from_gemini) # Print the whole response for inspection
    else:
        print("Search model not initialized. Skipping search execution.")
except IndexError:
    print("Error: No valid candidates or parts found in Gemini's response. This might indicate an issue with the prompt or model configuration.")
    if 'response_from_gemini' in globals(): print("Gemini's raw response was:", response_from_gemini)
except AttributeError as e:
    print(f"Error processing Gemini's response (AttributeError): {e}. This might indicate an unexpected response structure.")
    if 'response_from_gemini' in globals(): print("Gemini's raw response was:", response_from_gemini)
except Exception as e:
    print(f"An unexpected error occurred during the search phase: {e}")
    if 'response_from_gemini' in globals(): print("Gemini's raw response was:", response_from_gemini)

# At this point, `search_results_from_tool` holds the data retrieved by the search tool.
# The next cell will process this data using Gemini for extraction.

In [ ]:
# CELL 4: Implement Data Extraction and Structuring with Gemini

# --- Configuration: BigQuery Table Details ---
# TODO: Confirm or replace these with your desired BigQuery dataset and table IDs.
# The BIGQUERY_TABLE_SCHEMA defines the structure of your table in BigQuery.

BIGQUERY_DATASET_ID = "gemini_search_data" # <--- CONFIRM OR REPLACE: Your BigQuery Dataset ID
BIGQUERY_TABLE_ID = "technology_insights"  # <--- CONFIRM OR REPLACE: Your BigQuery Table ID

BIGQUERY_TABLE_SCHEMA = [
    bigquery.SchemaField("technology_name", "STRING", mode="NULLABLE", description="Name of the technology, product, or concept"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE", description="Concise description of the item"),
    bigquery.SchemaField("potential_impact", "STRING", mode="NULLABLE", description="Potential impact, applications, or significance"),
    bigquery.SchemaField("source_url", "STRING", mode="NULLABLE", description="URL of the source article/document"),
    bigquery.SchemaField("search_snippet", "STRING", mode="NULLABLE", description="The original snippet from the search result that provided this information")
]

print(f"BigQuery target table defined: {PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE_ID}")
print(f"Table Schema: {[(field.name, field.field_type) for field in BIGQUERY_TABLE_SCHEMA]}")

# --- Prepare Input for Gemini from Search Results ---
# The `search_results_from_tool` variable from the previous cell is used here.
# If it's empty or not available, placeholder data is used for demonstration.
if 'search_results_from_tool' not in globals() or not search_results_from_tool:
    print("\nWarning: `search_results_from_tool` is empty or not found. Using placeholder data for the extraction step.")
    # Placeholder data for demonstration if the search step failed or was skipped
    search_results_from_tool = [
        {'title': 'Breakthrough in AI-driven Material Science', 'link': 'http://example.com/ai_material', 'snippet': 'Scientists use AI to discover novel material with unique properties, potentially revolutionizing manufacturing.'},
        {'title': 'Next-Gen Quantum Entanglement Achieved', 'link': 'http://example.com/quantum_entanglement', 'snippet': 'Researchers demonstrate stable quantum entanglement over longer distances, a key step for quantum internet.'}
    ]

formatted_search_data_for_extraction = ""
for i, result in enumerate(search_results_from_tool):
    formatted_search_data_for_extraction += f"Source {i+1}:\nTitle: {result.get('title', 'N/A')}\nLink: {result.get('link', 'N/A')}\nSnippet: {result.get('snippet', 'N/A')}\n---\n"

# --- Initialize a new Generative Model for Extraction (or reuse) ---
# It's often cleaner to use a model instance without the search tool for pure text/JSON generation.
print("Initializing Gemini model for data extraction...")
try:
    extraction_model = GenerativeModel("gemini-1.5-pro-preview-0409") # Or "gemini-1.0-pro"
    print("Gemini model for extraction initialized successfully.")
except Exception as e:
    print(f"Error initializing Gemini model for extraction: {e}")

# --- Craft the Prompt for Gemini to Extract Structured Data ---
extraction_prompt_template = f"""
You are an expert data extraction specialist. Based on the following search result snippets, identify key technology breakthroughs or significant findings.
For each distinct item, extract the following information:
1.  `technology_name`: The name of the technology, product, concept, or key finding.
2.  `description`: A concise summary of what it is or what was discovered.
3.  `potential_impact`: A brief note on its potential applications, impact, or significance as suggested by the snippet.
4.  `source_url`: The URL of the article from which this information was derived.
5.  `search_snippet`: The original text snippet from the search result.

Format your response as a valid JSON list of objects. Each object in the list should strictly follow this structure:
{{ 
  "technology_name": "(name of technology/finding)",
  "description": "(summary of the technology/finding)",
  "potential_impact": "(potential impact or significance)",
  "source_url": "(full URL of the source article)",
  "search_snippet": "(original snippet text used for extraction)"
}}

If a piece of information is not available in a snippet for a specific field, use "Not specified" or null for that field's value.
Ensure the output is ONLY the JSON list, without any introductory text, explanations, or markdown formatting like ```json ... ```.

Here are the search result snippets to process:
{formatted_search_data_for_extraction}
"""

print("\n--- Sending prompt to Gemini for data extraction and structuring ---")
# print("Extraction Prompt (first 500 chars):", extraction_prompt_template[:500]) # For debugging prompt

extracted_structured_data = [] # Initialize to store the JSON output from Gemini

try:
    if 'extraction_model' in globals(): # Check if model was initialized
        extraction_response = extraction_model.generate_content(extraction_prompt_template)
        # print("Gemini's raw extraction response:", extraction_response.text) # For debugging
        
        response_text = extraction_response.text.strip()
        # Handle potential markdown ```json ... ``` wrapper from Gemini
        if response_text.startswith("```json") and response_text.endswith("```"):
            response_text = response_text[7:-3].strip()
        elif response_text.startswith("```") and response_text.endswith("```"): # More general markdown block
            response_text = response_text[3:-3].strip()

        extracted_structured_data = json.loads(response_text) # Parse the cleaned text to JSON
        print("\n--- Successfully extracted and parsed structured data from Gemini ---")
    else:
        print("Extraction model not initialized. Skipping data extraction.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON from Gemini's extraction response: {e}")
    if 'extraction_response' in locals(): print("Gemini's raw response was (check for malformed JSON or non-JSON text):\n", extraction_response.text)
except AttributeError:
    print("Error: Could not access 'text' attribute from Gemini's response. The response might be empty or malformed.")
    if 'extraction_response' in locals(): print("Gemini's raw response object:", extraction_response)
except Exception as e:
    print(f"An unexpected error occurred during data extraction: {e}")
    if 'extraction_response' in locals() and hasattr(extraction_response, 'text'): print("Gemini's raw response was:\n", extraction_response.text)

# --- Display the Extracted Structured Data ---
if extracted_structured_data:
    print(f"Successfully extracted {len(extracted_structured_data)} structured items.")
    for i, item in enumerate(extracted_structured_data):
        print(f"Item {i+1}: {item}")
else:
    print("\nNo structured data was extracted from the search results, or an error occurred in the process.")

# `extracted_structured_data` now holds the list of dictionaries, ready for BigQuery.

In [ ]:
# CELL 5: Load Data into BigQuery

# This cell handles creating the BigQuery dataset and table (if they don't exist)
# and then inserts the structured data extracted by Gemini in the previous cell.

print("\n--- Starting BigQuery Data Loading Operations ---")

# --- Verify Essential Variables ---
# Check if `bq_client` (BigQuery client) and other necessary variables are available.
essential_vars_present = True
for var_name in ['bq_client', 'PROJECT_ID', 'BIGQUERY_DATASET_ID', 'BIGQUERY_TABLE_ID', 'BIGQUERY_TABLE_SCHEMA', 'extracted_structured_data']:
    if var_name not in globals():
        print(f"Error: Essential variable '{var_name}' is not defined. Please ensure all previous cells have run successfully.")
        essential_vars_present = False
if not essential_vars_present:
    print("Cannot proceed with BigQuery operations due to missing variables. Halting.")
else:
    full_table_id = f"{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE_ID}"
    dataset_ref = bq_client.dataset(BIGQUERY_DATASET_ID) # Corrected way to get dataset reference
    table_ref = dataset_ref.table(BIGQUERY_TABLE_ID)

    try:
        # 1. Create BigQuery Dataset (if it doesn't exist)
        print(f"Ensuring BigQuery dataset '{BIGQUERY_DATASET_ID}' exists in project '{PROJECT_ID}'...")
        # dataset_ref = bigquery.DatasetReference(PROJECT_ID, BIGQUERY_DATASET_ID) # This is also valid
        bq_client.create_dataset(dataset_ref, exists_ok=True) # exists_ok=True prevents error if dataset already exists
        print(f"Dataset '{BIGQUERY_DATASET_ID}' is ready.")

        # 2. Create BigQuery Table (if it doesn't exist)
        print(f"Ensuring BigQuery table '{full_table_id}' exists with the defined schema...")
        table = bigquery.Table(table_ref, schema=BIGQUERY_TABLE_SCHEMA)
        bq_client.create_table(table, exists_ok=True) # exists_ok=True prevents error if table already exists
        print(f"Table '{full_table_id}' is ready.")

        # 3. Insert Data into the BigQuery Table
        if extracted_structured_data:
            # Filter out any None items if `extracted_structured_data` might contain them
            rows_to_insert = [row for row in extracted_structured_data if isinstance(row, dict)]
            
            if not rows_to_insert:
                print("No valid data (dictionaries) found in `extracted_structured_data` to insert.")
            else:
                print(f"Attempting to insert {len(rows_to_insert)} rows into '{full_table_id}'...")
                # Use insert_rows_json for list of dictionaries
                errors = bq_client.insert_rows_json(table_ref, rows_to_insert)
                if not errors: # If 'errors' list is empty, all rows were inserted successfully
                    print(f"Successfully inserted {len(rows_to_insert)} rows into '{full_table_id}'.")
                else:
                    print(f"Encountered errors while inserting rows into BigQuery. Details:")
                    for error_entry in errors:
                        print(f"  Row index: {error_entry.get('index', 'N/A')}")
                        print(f"  Errors: {error_entry.get('errors', 'N/A')}")
        else:
            print("No structured data was available (`extracted_structured_data` is empty). Nothing to insert into BigQuery.")

    except AttributeError as e:
        print(f"An AttributeError occurred during BigQuery operations: {e}.")
        print("This often happens if `bq_client` was not initialized correctly in Cell 2.")
        print("Please ensure the Vertex AI and BigQuery client initialization cell (Cell 2) has been run successfully and there were no errors.")
    except Exception as e:
        print(f"An unexpected error occurred during BigQuery operations: {e}")
        print("Check your BigQuery permissions, API enablement, and the correctness of dataset/table IDs.")

    print("\n--- BigQuery Data Loading Operations Finished ---")

    # Optional: Query the table to verify data insertion
    if essential_vars_present and extracted_structured_data: # Only try to query if setup was okay
        try:
            print(f"\n--- Verifying data by querying table: {full_table_id} (LIMIT 5) ---")
            query_job = bq_client.query(f"SELECT technology_name, source_url, description FROM `{full_table_id}` LIMIT 5")
            results = query_job.result() # Waits for the job to complete
            count = 0
            for row in results:
                print(f"  Row {count+1}: Name='{row.technology_name}', URL='{row.source_url}', Desc='{row.description[:50]}...' ")
                count += 1
            if count == 0:
                print("Query executed, but no rows returned. The table might be empty or the insert failed silently if not checked above.")
            print("Verification query finished.")
        except Exception as e:
            print(f"Error during verification query: {e}")

# Notebook Conclusion

This notebook has demonstrated a complete workflow:

1.  **Initialization**: Setting up Google Cloud clients for Vertex AI (Gemini) and BigQuery.
2.  **Search via Tool Use**: Prompting Gemini to use a custom `perform_google_search` function to find information based on a user query.
3.  **Data Extraction**: Using another Gemini prompt to parse the search results and extract structured information into a JSON format.
4.  **BigQuery Loading**: 
    *   Creating a BigQuery dataset and table (if they didn't already exist) based on a defined schema.
    *   Inserting the structured JSON data into the BigQuery table.
5.  **Verification**: A simple query was run to display a few rows from the BigQuery table.

## Next Steps

If all cells ran successfully, your data should now be available in the specified BigQuery table (`{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE_ID}`). You can:

*   **Query your data in BigQuery**: Go to the BigQuery console and run SQL queries against your new table.
*   **Analyze the data**: Use tools like Looker Studio, Google Sheets, or Python libraries (pandas, etc.) to connect to BigQuery and analyze the collected information.
*   **Expand the schema**: Modify `BIGQUERY_TABLE_SCHEMA` and the extraction prompt to capture more fields.
*   **Automate the pipeline**: This notebook can be converted into a script and scheduled to run periodically (e.g., using Google Cloud Functions, Cloud Run, or Vertex AI Pipelines) to keep your BigQuery table updated with new search information.
*   **Error Handling and Robustness**: For production use, enhance error handling, add retries for API calls, and implement more comprehensive logging.